In [ ]:
import pandas as pd
import numpy as np
import json

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Environment ready!")

Pandas: 2.2.3
NumPy: 2.4.6
Environment ready!


In [ ]:
# Phase 1.2 — Load FLAN-T5

!pip install -q transformers sentencepiece accelerate

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
CUDA available: True


In [ ]:
MODEL_ID = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Model loaded successfully!")
print("Model:", MODEL_ID)
print("Device:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!
Model: google/flan-t5-small
Device: cuda


In [ ]:
def generate_answer(question, context):

    prompt = f"""
    Answer the question using only the information
    provided in the context.

    Context: {context}

    Question: {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
context = """
Customers can return products within 30 days
of delivery.
"""

question = "What is the return period?"

response = generate_answer(question, context)

print("Question:", question)
print("Generated Response:", response)

Question: What is the return period?
Generated Response: 30 days


In [ ]:
prompt = """
Answer the question using only the information
provided in the context.

Context: Customers can return products within 30 days of delivery.

Question: What is the return period?

Answer:
"""

tokens = tokenizer(prompt, return_tensors="pt")

print("Input IDs:")
print(tokens["input_ids"])

print("\nToken count:")
print(tokens["input_ids"].shape[1])

print("\nAttention mask:")
print(tokens["attention_mask"])

Input IDs:
tensor([[11801,     8,   822,   338,   163,     8,   251,   937,    16,     8,
          2625,     5,  1193,  6327,    10, 16423,    54,  1205,   494,   441,
           604,   477,    13,  1929,     5, 11860,    10,   363,    19,     8,
          1205,  1059,    58, 11801,    10,     1]])

Token count:
36

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
token_ids = tokens["input_ids"][0]

token_pieces = tokenizer.convert_ids_to_tokens(
    token_ids
)

for token_id, token_piece in zip(token_ids, token_pieces):
    print(f"{token_id.item():8} -> {token_piece}")

   11801 -> ▁Answer
       8 -> ▁the
     822 -> ▁question
     338 -> ▁using
     163 -> ▁only
       8 -> ▁the
     251 -> ▁information
     937 -> ▁provided
      16 -> ▁in
       8 -> ▁the
    2625 -> ▁context
       5 -> .
    1193 -> ▁Con
    6327 -> text
      10 -> :
   16423 -> ▁Customers
      54 -> ▁can
    1205 -> ▁return
     494 -> ▁products
     441 -> ▁within
     604 -> ▁30
     477 -> ▁days
      13 -> ▁of
    1929 -> ▁delivery
       5 -> .
   11860 -> ▁Question
      10 -> :
     363 -> ▁What
      19 -> ▁is
       8 -> ▁the
    1205 -> ▁return
    1059 -> ▁period
      58 -> ?
   11801 -> ▁Answer
      10 -> :
       1 -> </s>


In [ ]:
print("Input tensor shape:", tokens["input_ids"].shape)

print("Input tensor dtype:", tokens["input_ids"].dtype)

print("Input tensor device:", tokens["input_ids"].device)

print("\nAttention mask shape:", tokens["attention_mask"].shape)

print("Attention mask sum:", tokens["attention_mask"].sum().item())

Input tensor shape: torch.Size([1, 36])
Input tensor dtype: torch.int64
Input tensor device: cpu

Attention mask shape: torch.Size([1, 36])
Attention mask sum: 36


In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

print("Before moving:")
print("Input device:", inputs["input_ids"].device)
print("Model device:", next(model.parameters()).device)

inputs = inputs.to(device)

print("\nAfter moving:")
print("Input device:", inputs["input_ids"].device)
print("Model device:", next(model.parameters()).device)

Before moving:
Input device: cpu
Model device: cuda:0

After moving:
Input device: cuda:0
Model device: cuda:0


In [ ]:
knowledge_base = {
    "returns": {
        "policy": "Customers can return products within 30 days of delivery."
    },

    "delivery": {
        "policy": "Standard delivery takes 3 to 5 business days."
    },

    "cancellation": {
        "policy": "Orders cannot be cancelled after they have been shipped."
    },

    "refunds": {
        "policy": "Refunds are processed within 5 to 7 business days after return approval."
    },

    "payment": {
        "policy": "Cash on delivery is available for orders below ₹5,000."
    },

    "exchange": {
        "policy": "Damaged products can be exchanged within 7 days of delivery."
    },

    "order_tracking": {
        "policy": "Customers can track orders using the tracking link sent to their registered email."
    },

    "international_delivery": {
        "policy": "The company currently delivers only within India."
    },

    "failed_payment": {
        "policy": "If payment fails, the order is not confirmed. Customers should retry payment or contact support if the amount was deducted."
    },

    "address_change": {
        "policy": "Delivery addresses can be changed before the order is shipped. Changes are not permitted after shipping."
    }
}

print("Knowledge Base created!")
print("Total policies:", len(knowledge_base))

Knowledge Base created!
Total policies: 10


In [ ]:
golden_test_case = {
    "test_id": "TC001",
    "category": "returns",
    "question": "What is the return period?",
    "context": knowledge_base["returns"]["policy"],
    "golden_answer": "Customers can return products within 30 days of delivery.",
    "evaluation_criteria": "The answer must correctly mention the 30-day return period."
}

for key, value in golden_test_case.items():
    print(f"{key}: {value}")

test_id: TC001
category: returns
question: What is the return period?
context: Customers can return products within 30 days of delivery.
golden_answer: Customers can return products within 30 days of delivery.
evaluation_criteria: The answer must correctly mention the 30-day return period.


In [ ]:
additional_test_cases = [
    {
        "test_id": "TC002",
        "category": "delivery",
        "question": "How long does standard delivery take?",
        "context": knowledge_base["delivery"]["policy"],
        "golden_answer": "Standard delivery takes 3 to 5 business days.",
        "evaluation_criteria": "Must mention 3 to 5 business days."
    },
    {
        "test_id": "TC003",
        "category": "cancellation",
        "question": "Can I cancel my order after shipping?",
        "context": knowledge_base["cancellation"]["policy"],
        "golden_answer": "No, orders cannot be cancelled after shipping.",
        "evaluation_criteria": "Must clearly state that shipped orders cannot be cancelled."
    },
    {
        "test_id": "TC004",
        "category": "refunds",
        "question": "When will I receive my refund?",
        "context": knowledge_base["refunds"]["policy"],
        "golden_answer": "Refunds are processed within 5 to 7 business days after return approval.",
        "evaluation_criteria": "Must mention 5 to 7 business days and return approval."
    },
    {
        "test_id": "TC005",
        "category": "payment",
        "question": "Is cash on delivery available?",
        "context": knowledge_base["payment"]["policy"],
        "golden_answer": "Yes, cash on delivery is available for orders below ₹5,000.",
        "evaluation_criteria": "Must mention COD availability and the ₹5,000 restriction."
    },
    {
        "test_id": "TC006",
        "category": "exchange",
        "question": "Can I exchange a damaged product?",
        "context": knowledge_base["exchange"]["policy"],
        "golden_answer": "Yes, damaged products can be exchanged within 7 days of delivery.",
        "evaluation_criteria": "Must mention damaged product eligibility and the 7-day period."
    },
    {
        "test_id": "TC007",
        "category": "order_tracking",
        "question": "How can I track my order?",
        "context": knowledge_base["order_tracking"]["policy"],
        "golden_answer": "You can track your order using the tracking link sent to your registered email.",
        "evaluation_criteria": "Must explain that the tracking link is sent to the registered email."
    },
    {
        "test_id": "TC008",
        "category": "international_delivery",
        "question": "Are international deliveries available?",
        "context": knowledge_base["international_delivery"]["policy"],
        "golden_answer": "No, deliveries are currently available only within India.",
        "evaluation_criteria": "Must clearly state that delivery is restricted to India."
    },
    {
        "test_id": "TC009",
        "category": "failed_payment",
        "question": "What happens if my payment fails?",
        "context": knowledge_base["failed_payment"]["policy"],
        "golden_answer": "Your order is not confirmed if payment fails. Retry the payment, or contact support if the amount was deducted.",
        "evaluation_criteria": "Must explain order confirmation, retrying payment and contacting support if money was deducted."
    },
    {
        "test_id": "TC010",
        "category": "address_change",
        "question": "Can I change my delivery address?",
        "context": knowledge_base["address_change"]["policy"],
        "golden_answer": "Yes, you can change your delivery address before shipping. Changes are not allowed after shipping.",
        "evaluation_criteria": "Must distinguish between pre-shipment and post-shipment address changes."
    }
]

print("Additional test cases created:", len(additional_test_cases))

Additional test cases created: 9


In [ ]:
import pandas as pd

all_test_cases = [golden_test_case] + additional_test_cases

golden_df = pd.DataFrame(all_test_cases)

print("Dataset shape:", golden_df.shape)
print("\nColumns:", golden_df.columns.tolist())

display(golden_df)

Dataset shape: (10, 6)

Columns: ['test_id', 'category', 'question', 'context', 'golden_answer', 'evaluation_criteria']


,test_id,category,question,context,golden_answer,evaluation_criteria
0,TC001,returns,What is the return period?,Customers can return products within 30 days o...,Customers can return products within 30 days o...,The answer must correctly mention the 30-day r...
1,TC002,delivery,How long does standard delivery take?,Standard delivery takes 3 to 5 business days.,Standard delivery takes 3 to 5 business days.,Must mention 3 to 5 business days.
2,TC003,cancellation,Can I cancel my order after shipping?,Orders cannot be cancelled after they have bee...,"No, orders cannot be cancelled after shipping.",Must clearly state that shipped orders cannot ...
3,TC004,refunds,When will I receive my refund?,Refunds are processed within 5 to 7 business d...,Refunds are processed within 5 to 7 business d...,Must mention 5 to 7 business days and return a...
4,TC005,payment,Is cash on delivery available?,Cash on delivery is available for orders below...,"Yes, cash on delivery is available for orders ...","Must mention COD availability and the ₹5,000 r..."
5,TC006,exchange,Can I exchange a damaged product?,Damaged products can be exchanged within 7 day...,"Yes, damaged products can be exchanged within ...",Must mention damaged product eligibility and t...
6,TC007,order_tracking,How can I track my order?,Customers can track orders using the tracking ...,You can track your order using the tracking li...,Must explain that the tracking link is sent to...
7,TC008,international_delivery,Are international deliveries available?,The company currently delivers only within India.,"No, deliveries are currently available only wi...",Must clearly state that delivery is restricted...
8,TC009,failed_payment,What happens if my payment fails?,"If payment fails, the order is not confirmed. ...",Your order is not confirmed if payment fails. ...,"Must explain order confirmation, retrying paym..."
9,TC010,address_change,Can I change my delivery address?,Delivery addresses can be changed before the o...,"Yes, you can change your delivery address befo...",Must distinguish between pre-shipment and post...


In [ ]:
print("Dataset shape:", golden_df.shape)

print("\nMissing values:")
print(golden_df.isnull().sum())

print("\nDuplicate test IDs:")
print(golden_df["test_id"].duplicated().sum())

print("\nUnique test IDs:")
print(golden_df["test_id"].nunique())

print("\nCategory distribution:")
print(golden_df["category"].value_counts())

Dataset shape: (10, 6)

Missing values:
test_id                0
category               0
question               0
context                0
golden_answer          0
evaluation_criteria    0
dtype: int64

Duplicate test IDs:
0

Unique test IDs:
10

Category distribution:
category
returns                   1
delivery                  1
cancellation              1
refunds                   1
payment                   1
exchange                  1
order_tracking            1
international_delivery    1
failed_payment            1
address_change            1
Name: count, dtype: int64


In [ ]:
import os
import json

os.makedirs("data", exist_ok=True)

# Save Knowledge Base
with open("data/knowledge_base.json", "w", encoding="utf-8") as file:
    json.dump(
        knowledge_base,
        file,
        indent=4,
        ensure_ascii=False
    )

# Save Golden Dataset
golden_df.to_csv(
    "data/golden_dataset.csv",
    index=False
)

print("Files saved successfully!")

print("\nSaved files:")
print(os.listdir("data"))

Files saved successfully!

Saved files:
['golden_dataset.csv', 'knowledge_base.json']


In [ ]:
def generate_answer(question, context):

    prompt = f"""
    Answer the question using only the information
    provided in the context.

    Context: {context}

    Question: {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
test_case = golden_df.iloc[0]

question = test_case["question"]
context = test_case["context"]

candidate_response = generate_answer(
    question,
    context
)

print("Test ID:", test_case["test_id"])
print("Question:", question)
print("Context:", context)
print("Candidate Response:", candidate_response)

Test ID: TC001
Question: What is the return period?
Context: Customers can return products within 30 days of delivery.
Candidate Response: 30 days


In [ ]:
candidate_responses = []

for _, row in golden_df.iterrows():

    response = generate_answer(
        row["question"],
        row["context"]
    )

    candidate_responses.append({
        "test_id": row["test_id"],
        "candidate_response": response
    })

candidate_df = pd.DataFrame(candidate_responses)

display(candidate_df)

,test_id,candidate_response
0,TC001,30 days
1,TC002,3 to 5 business days
2,TC003,No
3,TC004,within 5 to 7 business days
4,TC005,Yes
5,TC006,Yes
6,TC007,using the tracking link sent to their register...
7,TC008,yes
8,TC009,Customers should retry payment or contact supp...
9,TC010,Yes


In [ ]:
for _, row in candidate_df.iterrows():
    print(f"\nTest ID: {row['test_id']}")
    print(f"Response: {row['candidate_response']}")
    print("-" * 60)


Test ID: TC001
Response: 30 days
------------------------------------------------------------

Test ID: TC002
Response: 3 to 5 business days
------------------------------------------------------------

Test ID: TC003
Response: No
------------------------------------------------------------

Test ID: TC004
Response: within 5 to 7 business days
------------------------------------------------------------

Test ID: TC005
Response: Yes
------------------------------------------------------------

Test ID: TC006
Response: Yes
------------------------------------------------------------

Test ID: TC007
Response: using the tracking link sent to their registered email
------------------------------------------------------------

Test ID: TC008
Response: yes
------------------------------------------------------------

Test ID: TC009
Response: Customers should retry payment or contact support if the amount was deducted
------------------------------------------------------------

Test ID: TC0

In [ ]:
evaluation_df = golden_df.merge(
    candidate_df,
    on="test_id",
    how="inner"
)

print("Evaluation Dataset Shape:", evaluation_df.shape)

display(evaluation_df)

Evaluation Dataset Shape: (10, 7)


,test_id,category,question,context,golden_answer,evaluation_criteria,candidate_response
0,TC001,returns,What is the return period?,Customers can return products within 30 days o...,Customers can return products within 30 days o...,The answer must correctly mention the 30-day r...,30 days
1,TC002,delivery,How long does standard delivery take?,Standard delivery takes 3 to 5 business days.,Standard delivery takes 3 to 5 business days.,Must mention 3 to 5 business days.,3 to 5 business days
2,TC003,cancellation,Can I cancel my order after shipping?,Orders cannot be cancelled after they have bee...,"No, orders cannot be cancelled after shipping.",Must clearly state that shipped orders cannot ...,No
3,TC004,refunds,When will I receive my refund?,Refunds are processed within 5 to 7 business d...,Refunds are processed within 5 to 7 business d...,Must mention 5 to 7 business days and return a...,within 5 to 7 business days
4,TC005,payment,Is cash on delivery available?,Cash on delivery is available for orders below...,"Yes, cash on delivery is available for orders ...","Must mention COD availability and the ₹5,000 r...",Yes
5,TC006,exchange,Can I exchange a damaged product?,Damaged products can be exchanged within 7 day...,"Yes, damaged products can be exchanged within ...",Must mention damaged product eligibility and t...,Yes
6,TC007,order_tracking,How can I track my order?,Customers can track orders using the tracking ...,You can track your order using the tracking li...,Must explain that the tracking link is sent to...,using the tracking link sent to their register...
7,TC008,international_delivery,Are international deliveries available?,The company currently delivers only within India.,"No, deliveries are currently available only wi...",Must clearly state that delivery is restricted...,yes
8,TC009,failed_payment,What happens if my payment fails?,"If payment fails, the order is not confirmed. ...",Your order is not confirmed if payment fails. ...,"Must explain order confirmation, retrying paym...",Customers should retry payment or contact supp...
9,TC010,address_change,Can I change my delivery address?,Delivery addresses can be changed before the o...,"Yes, you can change your delivery address befo...",Must distinguish between pre-shipment and post...,Yes


In [ ]:
evaluation_df.to_csv(
    "data/evaluation_dataset.csv",
    index=False
)

print("Evaluation dataset saved successfully!")

Evaluation dataset saved successfully!


In [ ]:
!pip install -q laya

In [ ]:
import importlib.metadata

try:
    version = importlib.metadata.version("laya")
    print("Laya installed successfully!")
    print("Version:", version)

except importlib.metadata.PackageNotFoundError:
    print("Laya package is not installed.")

Laya installed successfully!
Version: 0.4.21


In [ ]:
import laya

print("Laya module location:")
print(laya.__file__)

print("\nAvailable public attributes:")

print([
    name for name in dir(laya)
    if not name.startswith("_")
])

Laya module location:
/usr/local/lib/python3.13/dist-packages/laya/__init__.py

Available public attributes:
['Agent', 'AsyncHook', 'BaseHook', 'DEFAULT_MODELS', 'DecisionResult', 'GATE_STATES', 'Hook', 'LayaDecision', 'LayaEvaluator', 'LayaGuardrail', 'LayaGuardrailError', 'LayaRouter', 'LayaTriage', 'OptionStabilityHook', 'PINNED_REVISIONS', 'PredictContext', 'PredictHook', 'QTYPES', 'QTYPE_NAMES', 'RLAgent', 'RouteDecision', 'Router', 'answer_confidence', 'apply_binning_map', 'apply_confidence_gate', 'cached_embed_fn', 'check_min_confidence', 'check_min_confidence_map', 'clean_email_body', 'confidence', 'confidence_from_probs', 'decide', 'decide_batch', 'detect_language', 'detect_script', 'ece_score', 'email', 'email_questions', 'email_state', 'embed_fn_from_agent', 'fit_abstention_thresholds', 'fit_binning_map', 'fit_one_temperature', 'fit_temperature_map', 'fit_temperatures', 'flag_low_confidence', 'guard_questions', 'hooks', 'is_english', 'jev_confidence', 'lang', 'load', 'modera

In [ ]:
agent = laya.load("convaiinnovations/laya")

print("Laya model loaded successfully!")
print("Agent type:", type(agent).__name__)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:2227: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Laya model loaded successfully!
Agent type: Agent


In [ ]:
import inspect

print("Available agent methods:\n")

for name in dir(agent):
    if not name.startswith("_"):
        attribute = getattr(agent, name)

        if callable(attribute):
            try:
                print(f"{name}{inspect.signature(attribute)}")
            except (TypeError, ValueError):
                print(name)

Available agent methods:

accelerate(use_graphs: bool = True, strict: bool = False)
add_hook(hook: 'HookArg') -> "'HookRegistry'"
deaccelerate()
decide(state: Union[str, dict, list], schema: Any = None, *, questions: Optional[Dict[str, Any]] = None, return_details: bool = False, min_confidence: Optional[float] = None, **predict_kwargs) -> Any
decide_batch(states: List[Union[str, dict, list]], schema: Any = None, *, questions: Optional[Dict[str, Any]] = None, return_details: bool = False, min_confidence: Optional[float] = None, **predict_kwargs) -> List[Any]
dtype_for(rows: int) -> torch.dtype
fit_binning(records, min_bucket_n: int = 200) -> Dict[str, Any]
fit_temperatures(records, compute_ece: bool = False, seed: int = 0) -> Dict[str, Any]
hooks_installed(*hooks: 'HookArg')
load_calibration(path: str) -> None
model(*args, **kwargs)
predict(state: Union[str, dict, list], questions: Dict[str, Dict[str, Any]], lang: Optional[str] = None, hooks=None, on_predict_start=None, on_predict_end=N

In [ ]:
# Select the first test case
test_case = evaluation_df.iloc[0]

# Prepare the information Laya needs to judge
state = {
    "question": test_case["question"],
    "context": test_case["context"],
    "golden_answer": test_case["golden_answer"],
    "evaluation_criteria": test_case["evaluation_criteria"],
    "candidate_response": test_case["candidate_response"]
}

# Define the evaluation rubric
evaluation_questions = {
    "answer_quality": {
        "type": "choice",
        "instructions": (
            "Compare the candidate response with the question, "
            "source context, golden answer and evaluation criteria. "
            "How well does the candidate response satisfy the criteria?"
        ),
        "criteria": {
            "meets_criteria": (
                "The response is correct and satisfies all important "
                "requirements in the evaluation criteria."
            ),
            "partially_meets_criteria": (
                "The response is partly correct but misses one or more "
                "important requirements."
            ),
            "does_not_meet_criteria": (
                "The response is incorrect, contradicts the context, "
                "or fails the important requirements."
            )
        }
    }
}

print("Evaluation state:")
print(state)

print("\nEvaluation rubric:")
print(evaluation_questions)

Evaluation state:
{'question': 'What is the return period?', 'context': 'Customers can return products within 30 days of delivery.', 'golden_answer': 'Customers can return products within 30 days of delivery.', 'evaluation_criteria': 'The answer must correctly mention the 30-day return period.', 'candidate_response': '30 days'}

Evaluation rubric:
{'answer_quality': {'type': 'choice', 'instructions': 'Compare the candidate response with the question, source context, golden answer and evaluation criteria. How well does the candidate response satisfy the criteria?', 'criteria': {'meets_criteria': 'The response is correct and satisfies all important requirements in the evaluation criteria.', 'partially_meets_criteria': 'The response is partly correct but misses one or more important requirements.', 'does_not_meet_criteria': 'The response is incorrect, contradicts the context, or fails the important requirements.'}}}


In [ ]:
judge_result = agent.predict(
    state=state,
    questions=evaluation_questions
)

print("Laya's evaluation:")
print(judge_result)

Laya's evaluation:
{'model': 'laya-rl-agent', 'answers': {'answer_quality': {'type': 'choice', 'choice': 'meets_criteria', 'probabilities': {'meets_criteria': 0.4503, 'partially_meets_criteria': 0.3019, 'does_not_meet_criteria': 0.2478}, 'confidence': 0.0292, 'answer_confidence': 0.4503, 'action': {'act_probability': 1.0}}}, 'usage': {'input_tokens': 174, 'output_tokens': 0, 'state_tokens': 73, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}}


In [ ]:
first_judgment = judge_result["answers"]["answer_quality"]

print("Test ID:", test_case["test_id"])
print("Candidate:", test_case["candidate_response"])
print("Laya decision:", first_judgment["choice"])
print("Confidence:", first_judgment["confidence"])
print("Probabilities:", first_judgment["probabilities"])

Test ID: TC001
Candidate: 30 days
Laya decision: meets_criteria
Confidence: 0.0292
Probabilities: {'meets_criteria': 0.4503, 'partially_meets_criteria': 0.3019, 'does_not_meet_criteria': 0.2478}


In [ ]:
judge_results = []

result_record = {
    "test_id": test_case["test_id"],
    "candidate_response": test_case["candidate_response"],
    "judge_decision": first_judgment["choice"],
    "judge_confidence": first_judgment["confidence"],
    "prob_meets": first_judgment["probabilities"]["meets_criteria"],
    "prob_partial": first_judgment["probabilities"]["partially_meets_criteria"],
    "prob_fails": first_judgment["probabilities"]["does_not_meet_criteria"]
}

judge_results.append(result_record)

print(pd.DataFrame(judge_results))

  test_id candidate_response  judge_decision  judge_confidence  prob_meets  \
0   TC001            30 days  meets_criteria            0.0292      0.4503   

   prob_partial  prob_fails  
0        0.3019      0.2478  


In [ ]:
for _, row in evaluation_df.iloc[1:].iterrows():

    state = {
        "question": row["question"],
        "context": row["context"],
        "golden_answer": row["golden_answer"],
        "evaluation_criteria": row["evaluation_criteria"],
        "candidate_response": row["candidate_response"]
    }

    result = agent.predict(
        state=state,
        questions=evaluation_questions
    )

    judgment = result["answers"]["answer_quality"]

    result_record = {
        "test_id": row["test_id"],
        "candidate_response": row["candidate_response"],
        "judge_decision": judgment["choice"],
        "judge_confidence": judgment["confidence"],
        "prob_meets": judgment["probabilities"]["meets_criteria"],
        "prob_partial": judgment["probabilities"]["partially_meets_criteria"],
        "prob_fails": judgment["probabilities"]["does_not_meet_criteria"]
    }

    judge_results.append(result_record)

judge_results_df = pd.DataFrame(judge_results)

print("Evaluation completed!")
print("Total test cases:", len(judge_results_df))

display(judge_results_df)

Evaluation completed!
Total test cases: 10


,test_id,candidate_response,judge_decision,judge_confidence,prob_meets,prob_partial,prob_fails
0,TC001,30 days,meets_criteria,0.0292,0.4503,0.3019,0.2478
1,TC002,3 to 5 business days,meets_criteria,0.1855,0.6439,0.2019,0.1542
2,TC003,No,does_not_meet_criteria,0.0620,0.3120,0.1946,0.4934
3,TC004,within 5 to 7 business days,meets_criteria,0.5096,0.8411,0.1062,0.0527
4,TC005,Yes,meets_criteria,0.3559,0.7616,0.1475,0.0909
5,TC006,Yes,meets_criteria,0.4629,0.8190,0.1178,0.0632
6,TC007,using the tracking link sent to their register...,meets_criteria,0.2789,0.7025,0.2061,0.0914
7,TC008,yes,meets_criteria,0.6308,0.8953,0.0632,0.0415
8,TC009,Customers should retry payment or contact supp...,meets_criteria,0.3828,0.7748,0.1482,0.0770
9,TC010,Yes,meets_criteria,0.3128,0.7377,0.1485,0.1138


In [ ]:
human_labels = {
    "TC001": "meets_criteria",
    "TC002": "meets_criteria",
    "TC003": "meets_criteria",
    "TC004": "partially_meets_criteria",
    "TC005": "partially_meets_criteria",
    "TC006": "partially_meets_criteria",
    "TC007": "meets_criteria",
    "TC008": "does_not_meet_criteria",
    "TC009": "partially_meets_criteria",
    "TC010": "partially_meets_criteria"
}

judge_results_df["human_label"] = (
    judge_results_df["test_id"].map(human_labels)
)

display(judge_results_df[
    ["test_id", "judge_decision", "human_label"]
])

,test_id,judge_decision,human_label
0,TC001,meets_criteria,meets_criteria
1,TC002,meets_criteria,meets_criteria
2,TC003,does_not_meet_criteria,meets_criteria
3,TC004,meets_criteria,partially_meets_criteria
4,TC005,meets_criteria,partially_meets_criteria
5,TC006,meets_criteria,partially_meets_criteria
6,TC007,meets_criteria,meets_criteria
7,TC008,meets_criteria,does_not_meet_criteria
8,TC009,meets_criteria,partially_meets_criteria
9,TC010,meets_criteria,partially_meets_criteria


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

labels = [
    "meets_criteria",
    "partially_meets_criteria",
    "does_not_meet_criteria"
]

y_true = judge_results_df["human_label"]
y_pred = judge_results_df["judge_decision"]

# Overall agreement
accuracy = accuracy_score(y_true, y_pred)

print("Human-Judge Agreement")
print("---------------------")
print(f"Accuracy: {accuracy:.2%}")

# Detailed metrics
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0
))

# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels
)

display(pd.DataFrame(
    cm,
    index=[f"Human: {label}" for label in labels],
    columns=[f"Laya: {label}" for label in labels]
))

Human-Judge Agreement
---------------------
Accuracy: 30.00%

Classification Report:
                          precision    recall  f1-score   support

          meets_criteria       0.33      0.75      0.46         4
partially_meets_criteria       0.00      0.00      0.00         5
  does_not_meet_criteria       0.00      0.00      0.00         1

                accuracy                           0.30        10
               macro avg       0.11      0.25      0.15        10
            weighted avg       0.13      0.30      0.18        10


Confusion Matrix:


,Laya: meets_criteria,Laya: partially_meets_criteria,Laya: does_not_meet_criteria
Human: meets_criteria,3,0,1
Human: partially_meets_criteria,5,0,0
Human: does_not_meet_criteria,1,0,0


In [ ]:
import inspect

print("Function signature:")
print(inspect.signature(laya.render_options))

print("\nFunction documentation:")
print(inspect.getdoc(laya.render_options))

Function signature:
(q: Dict) -> List[str]

Function documentation:
Render option texts in label-index order. Noul semantic order is always [false, true].


In [ ]:
rendered_options = laya.render_options(
    evaluation_questions["answer_quality"]
)

for index, option in enumerate(rendered_options):
    print(f"Option {index}: {option}")

KeyError: 't'

In [ ]:
import inspect

print(inspect.getsource(laya.render_options))

def render_options(q: Dict) -> List[str]:
    """Render option texts in label-index order. Noul semantic order is always [false, true]."""
    t, crit = q["t"], q.get("crit")
    if t != "noul" and "labels" in q:
        raise ValueError("labels is only supported for noul questions")
    if t == "choice":
        # only None/"" mean "no description"; 0 and False are legitimate criterion values.
        # `str(k)` unconditionally: a label with no description is rendered as itself, so an int
        # label used to come back as an int from a function annotated `-> List[str]` and then
        # reached `build_sequence`, which calls `.replace` on it and raised an AttributeError
        # naming neither the question nor the label. With a description the same label already
        # went through `"%s: %s" %` and was a str, which is why only the undescribed form broke.
        # `structured._enum_field` stringifies labels the same way; the returned answer still
        # carries the caller's 

In [ ]:
print(inspect.getsource(agent.predict))

    @torch.no_grad()
    def system_one(self, state: Union[str, dict, list], questions: Dict[str, Dict[str, Any]], lang: Optional[str] = None,
                   hooks=None, on_predict_start=None, on_predict_end=None,
                   hooks_raise: Optional[bool] = None,
                   hooks_timeout: Optional[float] = None,
                   max_len: Optional[int] = None,
                   head_max_len: Optional[int] = None,
                   min_confidence: Optional[float] = None) -> Dict[str, Any]:
        """Evaluate typed questions across state in a single, parallel forward pass.

        Args:
            state: Text string, JSON dict, or conversation turn list.
            questions: Dictionary mapping question_id -> question definition.
                - choice: {"type": "choice", "instructions": "...", "criteria": {"optA": "...", ...}}
                - score:  {"type": "score",  "instructions": "...", "criteria": ["lvl0", "lvl1", ...]}
                - noul:   {"type"

In [ ]:
print(inspect.getsource(agent.predict_batch))

    @torch.no_grad()
    @_reuse_question_tokens
    def predict_batch(self, states: List[Union[str, dict, list]], questions: Dict[str, Dict[str, Any]],
                      batch_size: Optional[int] = None, lang: Optional[str] = None,
                      hooks=None,
                      on_predict_start=None, on_predict_end=None,
                      hooks_raise: Optional[bool] = None,
                      hooks_timeout: Optional[float] = None,
                      max_len: Optional[int] = None,
                      head_max_len: Optional[int] = None,
                      sort_by_length: bool = False,
                      min_confidence: Optional[float] = None) -> List[Dict[str, Any]]:
        """Evaluate the same questions over many states, packing them into shared forward passes.

        This is the throughput path. `system_one`/`predict` handle one state per forward pass; on a
        GPU that leaves most of the batch dimension idle. `predict_batch` collates several stat

In [ ]:
print(inspect.getsource(agent._to_internal))

    @staticmethod
    def _to_internal(qdef: Dict) -> Dict:
        t = qdef["type"]
        crit = qdef.get("criteria")
        if t == "choice" and isinstance(crit, list):
            crit = {c: None for c in crit}
        elif t == "noul" and isinstance(crit, dict):
            # Normalize boolean literal keys to string keys ("true"/"false")
            crit = {str(k).lower(): v for k, v in crit.items()}
        ins = qdef["instructions"]
        if not isinstance(ins, str):
            # `ensure_ascii=False`, matching `serialize_state` and `render_criterion` in
            # common.py and the instructions path in shortlist.py. The default escaped
            # non-ASCII to literal `\uXXXX`, which the tokenizer then read as escape text:
            # on the English checkpoint one German question answered noul=0.1652 as a dict
            # and noul=0.2650 as the identical plain string.
            ins = json.dumps(ins, ensure_ascii=False)
        q = {"t": t, "ins": ins, "crit": cri

In [ ]:
import pandas as pd
import json
import os

policies = {
    "returns": [
        "Customers can return products within 30 days of delivery.",
        "Returned products must be unused and in their original packaging.",
        "Customers must provide the original order ID when requesting a return.",
        "Products marked as final sale cannot be returned.",
        "Return requests must be submitted through the customer account.",
        "Customers receive a return confirmation after approval.",
        "Products with missing accessories may be rejected for return.",
        "Customers can request a return for defective products within 30 days.",
        "Return eligibility is calculated from the delivery date.",
        "Customers are notified by email when a return request is rejected."
    ],

    "delivery": [
        "Standard delivery takes 3 to 5 business days.",
        "Express delivery takes 1 to 2 business days.",
        "Delivery timelines begin after the order is confirmed.",
        "Business days exclude Sundays and public holidays.",
        "Customers receive an estimated delivery date after dispatch.",
        "Delivery delays may occur due to weather conditions.",
        "Customers must provide a complete delivery address.",
        "Delivery charges are displayed before order confirmation.",
        "Orders placed after 6 PM are processed the next business day.",
        "A delivery attempt notification is sent to the registered phone number."
    ],

    "cancellation": [
        "Orders cannot be cancelled after they have been shipped.",
        "Customers can cancel orders before shipment.",
        "Cancellation requests must include the order ID.",
        "Cancelled prepaid orders are eligible for a refund.",
        "Cash-on-delivery orders are not charged when cancelled before dispatch.",
        "Customers receive an email after successful cancellation.",
        "Cancellation requests are subject to order processing status.",
        "Orders already handed to the delivery partner cannot be cancelled.",
        "Partial cancellation is available only when the order contains multiple items.",
        "Customers must contact support if the cancellation option is unavailable."
    ],

    "refunds": [
        "Refunds are processed within 5 to 7 business days after return approval.",
        "Refunds for prepaid orders are sent to the original payment method.",
        "Shipping charges are non-refundable unless the company made a delivery error.",
        "Customers receive an email when a refund is initiated.",
        "Refund processing time may vary by bank.",
        "Failed refunds must be reported to customer support.",
        "Cash-on-delivery refunds are transferred to the customer's bank account.",
        "Customers must provide valid bank details for cash-on-delivery refunds.",
        "Refunds are initiated only after returned products pass inspection.",
        "Customers can check refund status from their account."
    ],

    "payments": [
        "Cash on delivery is available for orders below ₹5,000.",
        "Credit cards are accepted for online payments.",
        "Debit cards are accepted for online payments.",
        "UPI payments are supported.",
        "Net banking is available for eligible banks.",
        "Payment must be completed before a prepaid order is confirmed.",
        "Customers should never share their payment PIN with support agents.",
        "The final payable amount is displayed before checkout confirmation.",
        "Promotional discounts are applied before the final payment amount is calculated.",
        "International credit cards are not currently accepted."
    ],

    "exchanges": [
        "Damaged products can be exchanged within 7 days of delivery.",
        "Exchange requests require the original order ID.",
        "Products must be available in stock for an exchange.",
        "Customers must upload photographs of damaged products when requested.",
        "Exchanges are subject to product inspection.",
        "Used products are not eligible for standard exchanges.",
        "Customers receive an email after exchange approval.",
        "Replacement products are shipped after exchange approval.",
        "Customers can request an exchange for a different size when available.",
        "Exchange requests submitted after 7 days are not eligible under the standard policy."
    ],

    "order_tracking": [
        "Customers can track orders using the tracking link sent to their registered email.",
        "Tracking information becomes available after dispatch.",
        "Customers receive a dispatch notification by email.",
        "Tracking updates may take up to 24 hours to appear.",
        "Customers can also find tracking details in their account.",
        "The tracking link displays the latest shipment status.",
        "Customers should contact support if tracking information is unavailable for 48 hours.",
        "Delivery partners provide shipment movement updates.",
        "Customers receive a delivery confirmation after successful delivery.",
        "Tracking links are unique to individual shipments."
    ],

    "order_modifications": [
        "Delivery addresses can be changed before the order is shipped.",
        "Changes are not permitted after shipping.",
        "Customers can update their phone number before order dispatch.",
        "Product quantities cannot be changed after order confirmation.",
        "Customers must contact support to request an address correction.",
        "Order modifications depend on processing status.",
        "Billing addresses can be updated before invoice generation.",
        "Customers cannot change the payment method after order confirmation.",
        "Product variants can be changed only before order processing begins.",
        "Customers receive confirmation when an order modification is successful."
    ],

    "warranty": [
        "Eligible electronic products include a 12-month manufacturer warranty.",
        "Warranty coverage begins on the date of delivery.",
        "Physical damage is not covered under the standard warranty.",
        "Customers must provide proof of purchase for warranty claims.",
        "Warranty claims are subject to product inspection.",
        "Consumable accessories are excluded from the standard warranty.",
        "Warranty service is available only at authorized service centres.",
        "Customers receive a service request number after claim registration.",
        "Unauthorized repairs may void warranty coverage.",
        "Warranty replacement depends on inspection and product availability."
    ],

    "customer_support": [
        "Customer support is available from Monday to Saturday.",
        "Customer support operates between 9 AM and 6 PM.",
        "Customers can contact support through email.",
        "Customers can contact support through the website help centre.",
        "Customers should include their order ID when contacting support about an order.",
        "Support requests receive a ticket reference number.",
        "Customers can check ticket status using the reference number.",
        "Complex support requests may require additional investigation.",
        "Customers should not share passwords or payment PINs with support agents.",
        "Support responses are sent to the registered email address."
    ]
}

# Convert the policy dictionary into structured records
knowledge_base_records = []

for category, category_policies in policies.items():

    for index, policy in enumerate(category_policies, start=1):

        knowledge_base_records.append({
            "policy_id": f"POL_{len(knowledge_base_records) + 1:03d}",
            "category": category,
            "policy": policy
        })

# Create DataFrame
kb_df = pd.DataFrame(knowledge_base_records)

# Validate the dataset
print("Knowledge Base Validation")
print("-------------------------")
print("Total records:", len(kb_df))
print("Unique policy IDs:", kb_df["policy_id"].nunique())
print("Missing values:", kb_df.isnull().sum().sum())
print("Duplicate policies:", kb_df["policy"].duplicated().sum())

print("\nCategory distribution:")
display(kb_df["category"].value_counts().to_frame("policy_count"))

# Save dataset
os.makedirs("data", exist_ok=True)

kb_df.to_csv(
    "data/knowledge_base.csv",
    index=False
)

with open("data/knowledge_base.json", "w", encoding="utf-8") as file:
    json.dump(
        knowledge_base_records,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nKnowledge Base saved successfully!")

Knowledge Base Validation
-------------------------
Total records: 100
Unique policy IDs: 100
Missing values: 0
Duplicate policies: 0

Category distribution:


,policy_count
category,
returns,10
delivery,10
cancellation,10
refunds,10
payments,10
exchanges,10
order_tracking,10
order_modifications,10
warranty,10



Knowledge Base saved successfully!


In [ ]:
import pandas as pd
import os

# Questions correspond to the 100 policies in our Knowledge Base.
questions_by_category = {

    "returns": [
        "How many days do customers have to return a product?",
        "What condition must a returned product satisfy?",
        "What information is required to request a return?",
        "Can customers return products marked as final sale?",
        "How can customers submit a return request?",
        "What happens after a return request is approved?",
        "Can a return be rejected if accessories are missing?",
        "How long do customers have to return defective products?",
        "From which date is return eligibility calculated?",
        "How are customers informed about rejected return requests?"
    ],

    "delivery": [
        "How long does standard delivery take?",
        "How long does express delivery take?",
        "When does the delivery timeline begin?",
        "Are Sundays and public holidays counted as business days?",
        "When do customers receive an estimated delivery date?",
        "What can cause delivery delays?",
        "What address information must customers provide?",
        "When are delivery charges displayed?",
        "What happens to orders placed after 6 PM?",
        "How are customers notified about a delivery attempt?"
    ],

    "cancellation": [
        "Can an order be cancelled after shipping?",
        "When can customers cancel their orders?",
        "What information must a cancellation request include?",
        "Are cancelled prepaid orders eligible for refunds?",
        "Are customers charged for COD orders cancelled before dispatch?",
        "How are customers notified about successful cancellation?",
        "What determines whether a cancellation request can be processed?",
        "Can an order be cancelled after it is handed to the delivery partner?",
        "When is partial order cancellation available?",
        "What should customers do if the cancellation option is unavailable?"
    ],

    "refunds": [
        "How long does a refund take after return approval?",
        "Where are refunds for prepaid orders sent?",
        "Are shipping charges refundable?",
        "How are customers notified when a refund is initiated?",
        "Why might refund processing times vary?",
        "What should customers do about a failed refund?",
        "How are COD refunds transferred?",
        "What information is required for a COD refund?",
        "When is a refund initiated after a product return?",
        "How can customers check their refund status?"
    ],

    "payments": [
        "What is the maximum order value for cash on delivery?",
        "Are credit cards accepted?",
        "Can customers pay using debit cards?",
        "Does the company support UPI payments?",
        "Is net banking available?",
        "When must payment be completed for a prepaid order?",
        "Should customers share their payment PIN with support agents?",
        "When can customers see the final payable amount?",
        "When are promotional discounts applied?",
        "Are international credit cards accepted?"
    ],

    "exchanges": [
        "How long do customers have to exchange damaged products?",
        "What information is needed for an exchange request?",
        "Can a product be exchanged if it is out of stock?",
        "What evidence might customers need to provide for damaged products?",
        "Are exchanges subject to product inspection?",
        "Can used products be exchanged under the standard policy?",
        "How are customers notified about exchange approval?",
        "When is a replacement product shipped?",
        "Can customers exchange a product for a different size?",
        "Can customers request an exchange after seven days?"
    ],

    "order_tracking": [
        "How can customers track their orders?",
        "When does tracking information become available?",
        "How are customers notified when an order is dispatched?",
        "How long can tracking updates take to appear?",
        "Where else can customers find their tracking details?",
        "What information does the tracking link display?",
        "What should customers do if tracking is unavailable for 48 hours?",
        "Who provides shipment movement updates?",
        "How are customers notified after successful delivery?",
        "Are tracking links shared between different shipments?"
    ],

    "order_modifications": [
        "When can customers change their delivery address?",
        "Can a delivery address be changed after shipping?",
        "When can customers update their phone number?",
        "Can product quantities be changed after order confirmation?",
        "How can customers request an address correction?",
        "What determines whether an order can be modified?",
        "When can customers update their billing address?",
        "Can customers change the payment method after order confirmation?",
        "When can customers change product variants?",
        "How are customers notified about successful order modifications?"
    ],

    "warranty": [
        "How long is the manufacturer warranty for eligible electronic products?",
        "When does warranty coverage begin?",
        "Is physical damage covered by the standard warranty?",
        "What proof is required for warranty claims?",
        "Are warranty claims subject to inspection?",
        "Are consumable accessories covered by the standard warranty?",
        "Where can customers obtain warranty service?",
        "What do customers receive after registering a warranty claim?",
        "Can unauthorized repairs affect warranty coverage?",
        "What determines whether a warranty replacement is possible?"
    ],

    "customer_support": [
        "On which days is customer support available?",
        "What are the customer support operating hours?",
        "Can customers contact support through email?",
        "Is there a website help centre?",
        "What should customers include when contacting support about an order?",
        "What do customers receive after submitting a support request?",
        "How can customers check their support ticket status?",
        "Why might some support requests take longer to resolve?",
        "Should customers share passwords or payment PINs with support agents?",
        "Where are support responses sent?"
    ]
}

# Create test cases using the existing Knowledge Base.
golden_records = []
test_number = 1

for category, category_questions in questions_by_category.items():

    category_policies = (
        kb_df[kb_df["category"] == category]
        .reset_index(drop=True)
    )

    assert len(category_policies) == len(category_questions), (
        f"Question-policy count mismatch: {category}"
    )

    for index, question in enumerate(category_questions):

        policy = category_policies.loc[index, "policy"]

        golden_records.append({
            "test_id": f"TC{test_number:03d}",
            "policy_id": category_policies.loc[index, "policy_id"],
            "category": category,
            "difficulty": (
                "easy" if index < 4
                else "medium" if index < 7
                else "challenging"
            ),
            "question": question,
            "context": policy,
            "golden_answer": policy,

            "correctness_criteria": (
                f"The answer must be factually correct and consistent "
                f"with this policy: {policy}"
            ),

            "relevance_criteria": (
                "The answer must directly address the question asked."
            ),

            "completeness_criteria": (
                f"The answer must include all important details, "
                f"conditions and restrictions in this policy: {policy}"
            ),

            "conciseness_criteria": (
                "The answer must communicate the required information "
                "without unnecessary repetition or unrelated details."
            ),

            "faithfulness_criteria": (
                f"Every factual claim must be supported by the supplied "
                f"context. The answer must not introduce unsupported "
                f"information. Source context: {policy}"
            )
        })

        test_number += 1


# Build the Golden Dataset
golden_df = pd.DataFrame(golden_records)

# Validate
print("GOLDEN DATASET VALIDATION")
print("=" * 40)

print("Total test cases:", len(golden_df))
print("Unique test IDs:", golden_df["test_id"].nunique())
print("Unique policy IDs:", golden_df["policy_id"].nunique())
print("Missing values:", golden_df.isnull().sum().sum())
print("Duplicate test IDs:", golden_df["test_id"].duplicated().sum())

print("\nCategory distribution:")
display(golden_df["category"].value_counts().to_frame("test_cases"))

print("\nDifficulty distribution:")
display(golden_df["difficulty"].value_counts().to_frame("test_cases"))

print("\nSample test cases:")
display(golden_df.head(5))

# Save
golden_df.to_csv(
    "data/golden_dataset.csv",
    index=False
)

print("\nGolden Dataset saved successfully!")

GOLDEN DATASET VALIDATION
Total test cases: 100
Unique test IDs: 100
Unique policy IDs: 100
Missing values: 0
Duplicate test IDs: 0

Category distribution:


,test_cases
category,
returns,10
delivery,10
cancellation,10
refunds,10
payments,10
exchanges,10
order_tracking,10
order_modifications,10
warranty,10



Difficulty distribution:


,test_cases
difficulty,
easy,40
medium,30
challenging,30



Sample test cases:


,test_id,policy_id,category,difficulty,question,context,golden_answer,correctness_criteria,relevance_criteria,completeness_criteria,conciseness_criteria,faithfulness_criteria
0,TC001,POL_001,returns,easy,How many days do customers have to return a pr...,Customers can return products within 30 days o...,Customers can return products within 30 days o...,The answer must be factually correct and consi...,The answer must directly address the question ...,"The answer must include all important details,...",The answer must communicate the required infor...,Every factual claim must be supported by the s...
1,TC002,POL_002,returns,easy,What condition must a returned product satisfy?,Returned products must be unused and in their ...,Returned products must be unused and in their ...,The answer must be factually correct and consi...,The answer must directly address the question ...,"The answer must include all important details,...",The answer must communicate the required infor...,Every factual claim must be supported by the s...
2,TC003,POL_003,returns,easy,What information is required to request a return?,Customers must provide the original order ID w...,Customers must provide the original order ID w...,The answer must be factually correct and consi...,The answer must directly address the question ...,"The answer must include all important details,...",The answer must communicate the required infor...,Every factual claim must be supported by the s...
3,TC004,POL_004,returns,easy,Can customers return products marked as final ...,Products marked as final sale cannot be returned.,Products marked as final sale cannot be returned.,The answer must be factually correct and consi...,The answer must directly address the question ...,"The answer must include all important details,...",The answer must communicate the required infor...,Every factual claim must be supported by the s...
4,TC005,POL_005,returns,medium,How can customers submit a return request?,Return requests must be submitted through the ...,Return requests must be submitted through the ...,The answer must be factually correct and consi...,The answer must directly address the question ...,"The answer must include all important details,...",The answer must communicate the required infor...,Every factual claim must be supported by the s...



Golden Dataset saved successfully!


In [ ]:
candidate_records = []

for _, row in golden_df.iterrows():

    answer = generate_answer(
        question=row["question"],
        context=row["context"]
    )

    candidate_records.append({
        "test_id": row["test_id"],
        "candidate_response": answer
    })

candidate_df = pd.DataFrame(candidate_records)

print("Candidate generation completed!")
print("Total responses:", len(candidate_df))

display(candidate_df.head(10))

candidate_df.to_csv(
    "data/candidate_responses.csv",
    index=False
)

print("Candidate responses saved!")

Candidate generation completed!
Total responses: 100


,test_id,candidate_response
0,TC001,30
1,TC002,i.
2,TC003,original order ID
3,TC004,No
4,TC005,through the customer account
5,TC006,Customers receive a return confirmation
6,TC007,yes
7,TC008,30 days
8,TC009,delivery date
9,TC010,by email


Candidate responses saved!


In [ ]:
evaluation_df = golden_df.merge(
    candidate_df,
    on="test_id",
    how="inner",
    validate="one_to_one"
)

print("Evaluation Dataset")
print("------------------")
print("Total records:", len(evaluation_df))
print("Columns:", evaluation_df.shape[1])
print("Missing values:", evaluation_df.isnull().sum().sum())

display(
    evaluation_df[
        [
            "test_id",
            "question",
            "golden_answer",
            "candidate_response"
        ]
    ].head(10)
)

evaluation_df.to_csv(
    "data/evaluation_dataset.csv",
    index=False
)

print("\nEvaluation dataset saved!")

Evaluation Dataset
------------------
Total records: 100
Columns: 13
Missing values: 0


,test_id,question,golden_answer,candidate_response
0,TC001,How many days do customers have to return a pr...,Customers can return products within 30 days o...,30
1,TC002,What condition must a returned product satisfy?,Returned products must be unused and in their ...,i.
2,TC003,What information is required to request a return?,Customers must provide the original order ID w...,original order ID
3,TC004,Can customers return products marked as final ...,Products marked as final sale cannot be returned.,No
4,TC005,How can customers submit a return request?,Return requests must be submitted through the ...,through the customer account
5,TC006,What happens after a return request is approved?,Customers receive a return confirmation after ...,Customers receive a return confirmation
6,TC007,Can a return be rejected if accessories are mi...,Products with missing accessories may be rejec...,yes
7,TC008,How long do customers have to return defective...,Customers can request a return for defective p...,30 days
8,TC009,From which date is return eligibility calculated?,Return eligibility is calculated from the deli...,delivery date
9,TC010,How are customers informed about rejected retu...,Customers are notified by email when a return ...,by email



Evaluation dataset saved!


In [ ]:
# Define five independent evaluation questions for Laya

evaluation_questions = {

    "correctness": {
        "type": "choice",
        "instructions": (
            "Evaluate factual accuracy only. Treat the supplied context "
            "as the authoritative source for this benchmark. Assess whether "
            "the factual claims in the candidate answer are accurate. "
            "Do not penalize missing details here; evaluate those under completeness."
        ),
        "criteria": {
            "pass": "All factual claims are accurate and consistent with the context.",
            "partial": "The main factual answer is correct but contains a minor factual imprecision.",
            "fail": "The main answer is factually incorrect or contradicts the context."
        }
    },

    "relevance": {
        "type": "choice",
        "instructions": (
            "Evaluate whether the candidate answer addresses the user's "
            "actual question. Focus only on relevance, not completeness or "
            "factual accuracy."
        ),
        "criteria": {
            "pass": "The answer directly addresses the question.",
            "partial": "The answer addresses part of the question but also includes some unrelated information.",
            "fail": "The answer does not address the question or discusses an unrelated topic."
        }
    },

    "completeness": {
        "type": "choice",
        "instructions": (
            "Evaluate whether the candidate answer contains all information "
            "required by the question and its completeness criteria. "
            "Check important conditions, restrictions, quantities and time periods."
        ),
        "criteria": {
            "pass": "All essential information required by the question is included.",
            "partial": "The main answer is present but one or more required details are missing.",
            "fail": "The answer omits the central information required to answer the question."
        }
    },

    "conciseness": {
        "type": "choice",
        "instructions": (
            "Evaluate the efficiency of the candidate answer. "
            "Do not reward shortness alone. Assess whether the answer "
            "communicates its information without unnecessary repetition "
            "or irrelevant elaboration."
        ),
        "criteria": {
            "pass": "The answer is appropriately brief and communicates its information efficiently.",
            "partial": "The answer contains some unnecessary wording or repetition but remains reasonably clear.",
            "fail": "Excessive irrelevant or repetitive wording makes the answer inefficient or difficult to understand."
        }
    },

    "faithfulness": {
        "type": "choice",
        "instructions": (
            "Evaluate whether every factual claim in the candidate answer "
            "is supported by the supplied context. Do not use outside "
            "knowledge. Distinguish supported claims from unsupported "
            "additions and contradictions."
        ),
        "criteria": {
            "pass": "All factual claims are supported by the supplied context.",
            "partial": "The main answer is supported but includes a minor unsupported detail.",
            "fail": "The answer contains materially unsupported claims or contradicts the supplied context."
        }
    }
}


# Prepare the 100 evaluation states

evaluation_states = []

for _, row in evaluation_df.iterrows():

    evaluation_states.append({
        "question": row["question"],
        "context": row["context"],
        "golden_answer": row["golden_answer"],
        "candidate_response": row["candidate_response"],

        "correctness_criteria": row["correctness_criteria"],
        "relevance_criteria": row["relevance_criteria"],
        "completeness_criteria": row["completeness_criteria"],
        "conciseness_criteria": row["conciseness_criteria"],
        "faithfulness_criteria": row["faithfulness_criteria"]
    })


print("Total evaluation states:", len(evaluation_states))
print("Evaluation dimensions:", len(evaluation_questions))
print("Total expected judgments:",
      len(evaluation_states) * len(evaluation_questions))

Total evaluation states: 100
Evaluation dimensions: 5
Total expected judgments: 500


In [ ]:
laya_batch_results = agent.predict_batch(
    states=evaluation_states,
    questions=evaluation_questions,
    batch_size=8
)

print("Total cases evaluated:", len(laya_batch_results))
print("\nFirst evaluation result:")
print(laya_batch_results[0])

Total cases evaluated: 100

First evaluation result:
{'model': 'laya-rl-agent', 'answers': {'correctness': {'type': 'choice', 'choice': 'pass', 'probabilities': {'pass': 0.6714, 'partial': 0.1871, 'fail': 0.1415}, 'confidence': 0.2192, 'answer_confidence': 0.6714, 'action': {'act_probability': 1.0}}, 'relevance': {'type': 'choice', 'choice': 'pass', 'probabilities': {'pass': 0.5588, 'partial': 0.2728, 'fail': 0.1684}, 'confidence': 0.1083, 'answer_confidence': 0.5588, 'action': {'act_probability': 1.0}}, 'completeness': {'type': 'choice', 'choice': 'pass', 'probabilities': {'pass': 0.5357, 'partial': 0.2902, 'fail': 0.1741}, 'confidence': 0.0918, 'answer_confidence': 0.5357, 'action': {'act_probability': 1.0}}, 'conciseness': {'type': 'choice', 'choice': 'pass', 'probabilities': {'pass': 0.5512, 'partial': 0.2428, 'fail': 0.206}, 'confidence': 0.0921, 'answer_confidence': 0.5512, 'action': {'act_probability': 1.0}}, 'faithfulness': {'type': 'choice', 'choice': 'pass', 'probabilities': 

In [ ]:
laya_records = []

for i, result in enumerate(laya_batch_results):

    row = {
        "test_id": evaluation_df.iloc[i]["test_id"]
    }

    for dimension in evaluation_questions:

        answer = result["answers"][dimension]
        probabilities = answer.get("probabilities", {})

        row[f"{dimension}_decision"] = answer.get("choice")
        row[f"{dimension}_confidence"] = answer.get("confidence")
        row[f"{dimension}_answer_confidence"] = answer.get("answer_confidence")

        row[f"{dimension}_prob_pass"] = probabilities.get("pass")
        row[f"{dimension}_prob_partial"] = probabilities.get("partial")
        row[f"{dimension}_prob_fail"] = probabilities.get("fail")

    row["input_tokens"] = result["usage"].get("input_tokens")
    row["truncated"] = result["usage"].get("truncated")

    laya_records.append(row)


laya_results_df = pd.DataFrame(laya_records)

# Merge with the original evaluation dataset
final_evaluation_df = evaluation_df.merge(
    laya_results_df,
    on="test_id",
    how="inner",
    validate="one_to_one"
)

# Save results
laya_results_df.to_csv(
    "data/laya_evaluation_results.csv",
    index=False
)

final_evaluation_df.to_csv(
    "data/final_evaluation_dataset.csv",
    index=False
)

print("Laya results shape:", laya_results_df.shape)
print("Final evaluation dataset shape:", final_evaluation_df.shape)

display(laya_results_df.head())

Laya results shape: (100, 33)
Final evaluation dataset shape: (100, 45)


,test_id,correctness_decision,correctness_confidence,correctness_answer_confidence,correctness_prob_pass,correctness_prob_partial,correctness_prob_fail,relevance_decision,relevance_confidence,relevance_answer_confidence,...,conciseness_prob_partial,conciseness_prob_fail,faithfulness_decision,faithfulness_confidence,faithfulness_answer_confidence,faithfulness_prob_pass,faithfulness_prob_partial,faithfulness_prob_fail,input_tokens,truncated
0,TC001,pass,0.2192,0.6714,0.6714,0.1871,0.1415,pass,0.1083,0.5588,...,0.2428,0.2060,pass,0.1467,0.6071,0.6071,0.2273,0.1656,1442,False
1,TC002,pass,0.2818,0.7141,0.7141,0.1757,0.1102,pass,0.3069,0.7272,...,0.2118,0.1589,pass,0.3326,0.7463,0.7463,0.1603,0.0933,1467,False
2,TC003,pass,0.2082,0.6609,0.6609,0.2002,0.1389,pass,0.1490,0.5966,...,0.3118,0.2118,pass,0.2386,0.6842,0.6842,0.1887,0.1271,1492,False
3,TC004,pass,0.2981,0.7294,0.7294,0.1404,0.1301,pass,0.2220,0.6742,...,0.1386,0.1486,pass,0.2479,0.6935,0.6935,0.1729,0.1337,1397,False
4,TC005,pass,0.5294,0.8527,0.8527,0.0888,0.0585,pass,0.6028,0.8828,...,0.0946,0.0682,pass,0.6109,0.8872,0.8872,0.0691,0.0436,1432,False


In [ ]:
dimensions = [
    "correctness",
    "relevance",
    "completeness",
    "conciseness",
    "faithfulness"
]

for dimension in dimensions:

    print(f"\n{dimension.upper()}")
    print("-" * 35)

    counts = laya_results_df[
        f"{dimension}_decision"
    ].value_counts()

    percentages = (
        laya_results_df[f"{dimension}_decision"]
        .value_counts(normalize=True) * 100
    )

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

    display(summary)


CORRECTNESS
-----------------------------------


,Count,Percentage
correctness_decision,,
pass,97,97.0
partial,2,2.0
fail,1,1.0



RELEVANCE
-----------------------------------


,Count,Percentage
relevance_decision,,
pass,96,96.0
partial,3,3.0
fail,1,1.0



COMPLETENESS
-----------------------------------


,Count,Percentage
completeness_decision,,
pass,96,96.0
partial,3,3.0
fail,1,1.0



CONCISENESS
-----------------------------------


,Count,Percentage
conciseness_decision,,
pass,97,97.0
partial,2,2.0
fail,1,1.0



FAITHFULNESS
-----------------------------------


,Count,Percentage
faithfulness_decision,,
pass,96,96.0
partial,3,3.0
fail,1,1.0


In [ ]:
attention_records = []

for dimension in dimensions:

    mask = laya_results_df[
        f"{dimension}_decision"
    ].isin(["partial", "fail"])

    selected = final_evaluation_df.loc[
        mask,
        ["test_id", "question", "context",
         "golden_answer", "candidate_response"]
    ].copy()

    selected["dimension"] = dimension
    selected["decision"] = laya_results_df.loc[
        mask, f"{dimension}_decision"
    ].values

    attention_records.append(selected)

attention_df = pd.concat(attention_records, ignore_index=True)

print("Judgments requiring attention:", len(attention_df))

display(attention_df)

Judgments requiring attention: 18


,test_id,question,context,golden_answer,candidate_response,dimension,decision
0,TC035,Why might refund processing times vary?,Refund processing time may vary by bank.,Refund processing time may vary by bank.,a).,correctness,partial
1,TC054,What evidence might customers need to provide ...,Customers must upload photographs of damaged p...,Customers must upload photographs of damaged p...,photos,correctness,partial
2,TC072,Can a delivery address be changed after shipping?,Changes are not permitted after shipping.,Changes are not permitted after shipping.,No,correctness,fail
3,TC035,Why might refund processing times vary?,Refund processing time may vary by bank.,Refund processing time may vary by bank.,a).,relevance,partial
4,TC037,How are COD refunds transferred?,Cash-on-delivery refunds are transferred to th...,Cash-on-delivery refunds are transferred to th...,Cash-on-delivery refunds,relevance,partial
5,TC054,What evidence might customers need to provide ...,Customers must upload photographs of damaged p...,Customers must upload photographs of damaged p...,photos,relevance,partial
6,TC072,Can a delivery address be changed after shipping?,Changes are not permitted after shipping.,Changes are not permitted after shipping.,No,relevance,fail
7,TC035,Why might refund processing times vary?,Refund processing time may vary by bank.,Refund processing time may vary by bank.,a).,completeness,partial
8,TC037,How are COD refunds transferred?,Cash-on-delivery refunds are transferred to th...,Cash-on-delivery refunds are transferred to th...,Cash-on-delivery refunds,completeness,partial
9,TC054,What evidence might customers need to provide ...,Customers must upload photographs of damaged p...,Customers must upload photographs of damaged p...,photos,completeness,partial


In [ ]:
flagged_ids = ["TC035", "TC037", "TC054", "TC072", "TC084"]

probability_columns = [
    "test_id"
]

for dimension in dimensions:
    probability_columns.extend([
        f"{dimension}_decision",
        f"{dimension}_prob_pass",
        f"{dimension}_prob_partial",
        f"{dimension}_prob_fail"
    ])

display(
    laya_results_df[
        laya_results_df["test_id"].isin(flagged_ids)
    ][probability_columns]
)

,test_id,correctness_decision,correctness_prob_pass,correctness_prob_partial,correctness_prob_fail,relevance_decision,relevance_prob_pass,relevance_prob_partial,relevance_prob_fail,completeness_decision,...,completeness_prob_partial,completeness_prob_fail,conciseness_decision,conciseness_prob_pass,conciseness_prob_partial,conciseness_prob_fail,faithfulness_decision,faithfulness_prob_pass,faithfulness_prob_partial,faithfulness_prob_fail
34,TC035,partial,0.2313,0.4770,0.2917,partial,0.2211,0.4116,0.3673,partial,...,0.5136,0.2483,partial,0.2158,0.4367,0.3475,partial,0.2634,0.5157,0.2209
36,TC037,pass,0.4335,0.3089,0.2576,partial,0.3401,0.3870,0.2729,partial,...,0.3862,0.2316,pass,0.3555,0.3553,0.2891,partial,0.3399,0.3850,0.2750
53,TC054,partial,0.3556,0.4046,0.2398,partial,0.2942,0.4481,0.2576,partial,...,0.4344,0.2216,partial,0.3180,0.4364,0.2456,pass,0.4097,0.3776,0.2126
71,TC072,fail,0.2498,0.2333,0.5169,fail,0.2666,0.3288,0.4046,fail,...,0.2825,0.5072,fail,0.3101,0.2786,0.4113,fail,0.2392,0.2741,0.4867
83,TC084,pass,0.6364,0.2361,0.1274,pass,0.4875,0.3233,0.1892,pass,...,0.4006,0.1922,pass,0.3856,0.3770,0.2374,partial,0.3494,0.4381,0.2125


In [ ]:
import os

for file in os.listdir("data"):
    print(file)

evaluation_dataset.csv
final_evaluation_dataset.csv
knowledge_base.csv
laya_evaluation_results.csv
candidate_responses.csv
golden_dataset.csv
knowledge_base.json


In [ ]:
import os
import shutil

project_dir = "offline_llm_evaluation"

os.makedirs(f"{project_dir}/data", exist_ok=True)

files_to_copy = [
    "knowledge_base.csv",
    "final_evaluation_dataset.csv",
    "laya_evaluation_results.csv",
    "golden_dataset.csv",
    "candidate_responses.csv",
    "evaluation_dataset.csv",
    "knowledge_base.json"
]

for file in files_to_copy:
    shutil.copy(
        f"data/{file}",
        f"{project_dir}/data/{file}"
    )

print("Project structure created successfully!")

for root, dirs, files in os.walk(project_dir):
    print(root, files)

Project structure created successfully!
offline_llm_evaluation []
offline_llm_evaluation/data ['evaluation_dataset.csv', 'final_evaluation_dataset.csv', 'knowledge_base.csv', 'laya_evaluation_results.csv', 'candidate_responses.csv', 'golden_dataset.csv', 'knowledge_base.json']


In [ ]:
import shutil
from google.colab import files

# Compress the data folder
shutil.make_archive(
    "offline_llm_data",
    "zip",
    "offline_llm_evaluation/data"
)

# Download ZIP to your computer
files.download("offline_llm_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>